# E04 - SO SÁNH BỘ TÁCH TỪ CHO PHOBERT: `pyvi` VS `RDRSegmenter (VnCoreNLP)`

**Câu hỏi (từ bài review mục 5):**
`vinai/phobert-base-v2` được pretrain trên văn bản tách từ bằng **RDRSegmenter (VnCoreNLP)**, nhưng pipeline hiện dùng **pyvi**. Sự lệch ranh giới từ này sinh ra bao nhiêu subword thừa / `<unk>` và có làm giảm Dev F1 không?

**Tương thích:** chạy được trên **Local**, **Google Colab**, và **Kaggle GPU**.

In [3]:
import os, sys, glob, zipfile, subprocess

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')

def _find_root():
    cwd = os.getcwd()
    cand = os.path.abspath('..') if os.path.basename(cwd) in ('notebooks', 'experiments') else cwd
    if os.path.exists(os.path.join(cand, 'src', 'modeling.py')):
        return cand, cand

    if IN_COLAB:
        for base in [cwd, '/content']:
            if os.path.exists(os.path.join(base, 'src', 'modeling.py')):
                return base, base
        for z in glob.glob('/content/*src_and_data*.zip'):
            with zipfile.ZipFile(z, 'r') as zf:
                zf.extractall('/content')
            if os.path.exists('/content/src/modeling.py'):
                return '/content', '/content'

    if IN_KAGGLE:
        work = '/kaggle/working'
        if os.path.exists(os.path.join(work, 'src', 'modeling.py')):
            return work, work
        for p in glob.glob('/kaggle/input/**/src/modeling.py', recursive=True):
            data_root = os.path.dirname(os.path.dirname(p))
            return data_root, work
        for z in glob.glob('/kaggle/input/**/*src_and_data*.zip', recursive=True):
            dest = os.path.join(work, 'project')
            os.makedirs(dest, exist_ok=True)
            with zipfile.ZipFile(z, 'r') as zf:
                zf.extractall(dest)
            if os.path.exists(os.path.join(dest, 'src', 'modeling.py')):
                return dest, dest

    raise FileNotFoundError('Khong tim thay src/modeling.py! Hay chay trong repo hoac upload/attach src_and_data.zip.')

ROOT, WORK_DIR = _find_root()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
PROC_DIR = os.path.join(ROOT, 'data', 'processed')
os.environ['EXP_LOG_PATH'] = os.path.join(WORK_DIR, 'reports', 'experiments_log.csv')

for pkg in ['transformers', 'accelerate', 'scikit-learn']:
    mod = 'sklearn' if pkg == 'scikit-learn' else pkg
    try:
        __import__(mod)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print(f'Env: colab={IN_COLAB}, kaggle={IN_KAGGLE} | ROOT={ROOT} | WORK_DIR={WORK_DIR}')

Env: colab=True, kaggle=False | ROOT=/content | WORK_DIR=/content


In [4]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import time, shutil
import numpy as np
import pandas as pd
from transformers import AutoTokenizer
from src.modeling import PII_TOKENS
from src.exp_log import log_experiment

In [6]:
df_train = pd.read_csv(os.path.join(PROC_DIR, 'train_clean.csv'))
df_dev = pd.read_csv(os.path.join(PROC_DIR, 'dev_clean.csv'))
tok = AutoTokenizer.from_pretrained('vinai/phobert-base-v2')
tok.add_special_tokens({'additional_special_tokens': PII_TOKENS})

def audit_tokenizer(texts, name):
    unk_id = tok.unk_token_id
    bpe_lens, unk_cnt, tok_cnt = [], 0, 0
    for s in texts.fillna(''):
        ids = tok.encode(str(s), add_special_tokens=False)
        bpe_lens.append(len(ids))
        unk_cnt += sum(1 for i in ids if i == unk_id)
        tok_cnt += len(ids)
    arr = np.array(bpe_lens)
    return {
        'variant': name,
        'mean_bpe': round(float(arr.mean()), 2),
        'p99_bpe': int(np.quantile(arr, 0.99)),
        'max_bpe': int(arr.max()),
        'unk_rate_%': round(100.0 * unk_cnt / max(tok_cnt, 1), 4),
    }

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

## 1. SO SÁNH TỶ LỆ `<unk>` VÀ ĐỘ DÀI BPE TRÊN DEV

In [7]:
rows = [
    audit_tokenizer(df_dev['sentence_clean'], 'no_seg (sentence_clean)'),
    audit_tokenizer(df_dev['sentence_seg'], 'pyvi (sentence_seg)'),
]

rdr_ready = False
try:
    try:
        import py_vncorenlp
    except ImportError:
        if IN_COLAB or IN_KAGGLE:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'py_vncorenlp'])
            import py_vncorenlp
        else:
            raise
    assert shutil.which('java') is not None, 'May chua co Java (JDK/JRE) de chay VnCoreNLP.'
    vncore_dir = os.path.join(WORK_DIR, '.tmp_run', 'vncorenlp')
    os.makedirs(vncore_dir, exist_ok=True)
    if not os.path.exists(os.path.join(vncore_dir, 'VnCoreNLP-1.2.jar')):
        py_vncorenlp.download_model(save_dir=vncore_dir)
    rdr = py_vncorenlp.VnCoreNLP(annotators=['wseg'], save_dir=vncore_dir)
    df_train['sentence_rdr'] = df_train['sentence_clean'].fillna('').apply(lambda s: ' '.join(rdr.word_segment(s)))
    df_dev['sentence_rdr'] = df_dev['sentence_clean'].fillna('').apply(lambda s: ' '.join(rdr.word_segment(s)))
    rows.append(audit_tokenizer(df_dev['sentence_rdr'], 'RDRSegmenter (VnCoreNLP)'))
    rdr_ready = True
except Exception as e:
    print('Chua chay duoc RDRSegmenter:', e)

pd.DataFrame(rows)

,variant,mean_bpe,p99_bpe,max_bpe,unk_rate_%
0,no_seg (sentence_clean),42.67,129,249,0.1288
1,pyvi (sentence_seg),37.67,112,243,0.0383
2,RDRSegmenter (VnCoreNLP),37.30,112,241,0.0145


## 2. SO SÁNH PHOBERT DEV F1 (CHẠY KHI CÓ GPU & RDRSEGMENTER)

In [8]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
from src.modeling import ViSFDDataset, PhoBertMultiTask, dynamic_collate_fn, init_semantic_pii_embeddings

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '| rdr_ready:', rdr_ready)

if not rdr_ready:
    print('Bo qua buoc fine-tune vi chua co RDRSegmenter.')
else:
    asp_cols = [f'asp_{i}' for i in range(5)]
    pin = (device.type == 'cuda')

    def quick_train_eval(col_name, epochs=3, lr=2e-5, max_len=128):
        torch.manual_seed(42)
        tr_ds = ViSFDDataset(df_train[col_name].fillna(''), tok, df_train['sentiment'], df_train[asp_cols].values, max_len=max_len)
        dv_ds = ViSFDDataset(df_dev[col_name].fillna(''), tok, df_dev['sentiment'], df_dev[asp_cols].values, max_len=max_len)
        tr_dl = DataLoader(tr_ds, batch_size=32, shuffle=True, pin_memory=pin, collate_fn=lambda b: dynamic_collate_fn(b, tok.pad_token_id))
        dv_dl = DataLoader(dv_ds, batch_size=64, shuffle=False, pin_memory=pin, collate_fn=lambda b: dynamic_collate_fn(b, tok.pad_token_id))
        model = PhoBertMultiTask('vinai/phobert-base-v2', dropout=0.3).to(device)
        init_semantic_pii_embeddings(model.encoder, tok)
        model.to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        loss_s, loss_a = nn.CrossEntropyLoss(), nn.BCEWithLogitsLoss()
        t0 = time.time()
        for _ in range(epochs):
            model.train()
            for b in tr_dl:
                opt.zero_grad()
                ls, la = model(b['input_ids'].to(device), b['attention_mask'].to(device))
                loss = loss_s(ls, b['sentiment'].to(device)) + 0.9 * loss_a(la, b['aspects'].to(device))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
        model.eval()
        ps, pa = [], []
        with torch.no_grad():
            for b in dv_dl:
                ls, la = model(b['input_ids'].to(device), b['attention_mask'].to(device))
                ps.append(ls.argmax(-1).cpu().numpy())
                pa.append((la >= 0).int().cpu().numpy())
        sf1 = f1_score(df_dev['sentiment'], np.concatenate(ps), average='macro')
        af1 = f1_score(df_dev[asp_cols].values, np.vstack(pa), average='macro')
        log_experiment('E04', col_name, 'PhoBERT segmenter comparison',
                       {'col': col_name, 'epochs': epochs, 'lr': lr, 'max_len': max_len},
                       dev_f1=(sf1 + af1) / 2, notes=f'sent={sf1:.4f}, asp={af1:.4f}, sec={time.time()-t0:.1f}')
        return {'col': col_name, 'sent_f1': round(sf1, 4), 'asp_f1': round(af1, 4), 'mean_f1': round((sf1 + af1)/2, 4)}

    res = [quick_train_eval('sentence_seg'), quick_train_eval('sentence_rdr')]
    print(pd.DataFrame(res))

Device: cuda | rdr_ready: True


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  540MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


model.safetensors: reconstructing file:   0%|          |  0.00B /  540MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


            col  sent_f1  asp_f1  mean_f1
0  sentence_seg   0.8002  0.8991   0.8496
1  sentence_rdr   0.7956  0.9058   0.8507


## 3. KẾT LUẬN

- **Tỷ lệ `<unk>` trên dev:** `no_seg` = `0.1288%` -> `pyvi` = `0.0383%` -> `RDRSegmenter` = `0.0145%` (giảm hơn một nửa so với `pyvi`).
- **PhoBERT Dev F1 (3 epochs, lr=2e-5, max_len=128):** `pyvi` đạt `0.8496` (sent `0.8002`, asp `0.8991`), `RDRSegmenter` đạt `0.8507` (sent `0.7956`, asp `0.9058`).
- **Chốt:** Chênh lệch trung bình chỉ `+0.0011` nên nhóm giữ nguyên `pyvi` trong pipeline chính cho gọn nhẹ (không phụ thuộc Java/VnCoreNLP).